In [1]:
import json
import numpy as np
import pandas as pd
from langdetect import detect, detect_langs
from copy import deepcopy
import unicodedata
import ast

In [2]:
with open("data/all_results.json", "r", encoding="utf-8") as f:
            survey_full_results = json.load(f)

In [3]:
# 1. Setup the Master Answer Key
# Combining both languages into one set for high-speed, order-independent lookup
attn_chk_pass_answer_DK = [
    'En person, som ikke er fra et politisk parti, men støtter et parti.',
    "En person fra NGO'er, der fremmer valgdeltagelse."
]
attn_chk_pass_answer_EN = [
    'Someone who is not from a political party but supports a party.',
    'Someone from NGOs promoting electoral participation.'
]
valid_answers = set(attn_chk_pass_answer_DK + attn_chk_pass_answer_EN)

In [4]:
# 2. Initialize counters and lists
fail_count = 0
pass_count = 0
no_pass_list = []

# 3. Process the results
for res in survey_full_results:
    # Safely retrieve the list of answers from the nested dictionary
    attn_chk_res = res['surveyData']['votingExperience']['ATTN_CHK']
    if detect(attn_chk_res[0]) == 'en':  # output: 'en' for english; 'da' for danish
        res['surveyData']['language'] = 'EN'
    else: 
        res['surveyData']['language'] = 'DA'
    
    # Calculate overlap using set intersection
    # This finds which of the respondent's answers are in our 'valid_answers' set
    correct_selections = set(attn_chk_res) & valid_answers
    num_correct = len(correct_selections)
    total_selected = len(attn_chk_res)

    # 1. PASS: Exactly 2 correct and ONLY 2 selected.
    if num_correct == 2 and total_selected == 2:
        res['ATTN_CHK_PF'] = 'PASS'
        res['ATTN_CHK_FAIL_REASON'] = None
        pass_count += 1
        
    # 2. HALF_PASS: Picked 1 or 2 correct, but total selections must be less than 4.
    # This keeps people who picked (Correct + Incorrect) or (Correct + Correct + Incorrect).
    elif num_correct >= 1 and total_selected < 4:
        res['ATTN_CHK_PF'] = 'HALF_PASS'
        res['ATTN_CHK_FAIL_REASON'] = None
        no_pass_list.append(attn_chk_res)
        
    # 3. FAIL: Either zero correct OR they "gamed" the system by picking 4+ options.
    else:
        res['ATTN_CHK_PF'] = 'FAIL'
        # Optional: Add a specific reason for your internal tracking
        if total_selected >= 4:
            res['ATTN_CHK_FAIL_REASON'] = 'Over-selection (4+ items)'
        else:
            res['ATTN_CHK_FAIL_REASON'] = 'Zero correct matches'
            
        no_pass_list.append(attn_chk_res)
        fail_count += 1

# 4. Optional: Print summary
print(f"Results processed: {pass_count} Pass, {fail_count} Fail.")

Results processed: 557 Pass, 395 Fail.


In [5]:
column_name_list = []
initial_survey_data = survey_full_results[0]['surveyData']
for col_nam in initial_survey_data.keys():
    if type(initial_survey_data[col_nam]) is dict:
        column_name_list = column_name_list + list(initial_survey_data[col_nam].keys())
    else: column_name_list.append(col_nam)
column_name_list = column_name_list + list(survey_full_results[0].keys())[-2:]

In [6]:
translation_dict = {
    'SCREEN_ELIGIBILITY': {"EN": ['Yes'],
              "DA": ['Ja']},
    'SCREEN_RESIDENCE': {"EN": ['Yes'],
              "DA": ['Ja']},
    'ALCL11': {"EN": ['Municipal tax in Aarhus Municipality must be raised, and the money must be spent on better welfare.', 'It is possible to save money in the public sector without affecting public welfare.',
                     'More tasks in the public sector must be solved by private companies.', 'Aarhus Municipality must make it cheaper to run a business.',
                     'Aarhus Municipality must prioritize that school pupils are mixed according to ethnicity and social background.', 'The politicians must prevent the construction of mosques.',
                     'More parking spaces must be established in Aarhus Municipality.', 'The city council’s temporary stop for the expansion of Aarhus Harbor must be made permanent.',
                     'Car traffic in Aarhus city center must be limited, for example through one-way directions, speed reductions and a zero-emission zone.',
                     'The municipality must continue with the plans for the new football stadium in Kongelunden, even if the costs rise again.'],
              "DA": ['Kommuneskatten i Aarhus Kommune skal hæves, og pengene skal bruges på bedre velfærd.', 'Det er muligt at spare penge i den offentlige sektor uden at påvirke den offentlige velfærd.',
                     'Flere opgaver i den offentlige sektor skal løses af private virksomheder.', 'Aarhus Kommune skal gøre det billigere at drive virksomhed.',
                     'Aarhus Kommune skal prioritere, at skoleelever blandes på tværs af etnicitet og social baggrund.', 'Politikerne skal forhindre opførelsen af moskéer.',
                     'Der skal etableres flere parkeringspladser i Aarhus Kommune.', 'Byrådets midlertidige stop for udvidelsen af Aarhus Havn skal gøres permanent.',
                     'Biltrafikken i Aarhus centrum skal begrænses, for eksempel gennem ensretninger, hastighedsnedsættelser og en nulemissionszone.',
                     'Kommunen skal fortsætte planerne om det nye fodboldstadion i Kongelunden, selv hvis omkostningerne stiger igen.']},
    'VOTE_LOC': {"EN": ['I did not vote.', 'I thought about voting this time, but I didn’t.', 'I usually vote but didn’t this time.', 'I am sure I voted.'],
              "DA": ['Jeg stemte ikke.', 'Jeg overvejede at stemme denne gang, men jeg gjorde det ikke.', 
                     'Jeg plejer at stemme, men gjorde det ikke denne gang.', 'Jeg er sikker på, at jeg stemte.']},
    'VOTE_NAT': {"EN": ['I did not vote.', 'I thought about voting this time, but I didn’t.', 'I usually vote but didn’t this time.', 'I am sure I voted.'],
              "DA": ['Jeg stemte ikke.', 'Jeg overvejede at stemme denne gang, men jeg gjorde det ikke.', 
                     'Jeg plejer at stemme, men gjorde det ikke denne gang.', 'Jeg er sikker på, at jeg stemte.']},
    'ATTN_CHK': {"EN": ['Someone from a political party.', 'Someone who is not from a political party but supports a party.',
                       'Someone from NGOs promoting electoral participation.', 'No one from a political party.'],
              "DA": ['En person fra et politisk parti.', 'En person, som ikke er fra et politisk parti, men støtter et parti.',
                    "En person fra NGO'er, der fremmer valgdeltagelse.", 'Ingen fra et politisk parti.']},
    'GENDER': {"EN": ['Male', 'Female', 'Non-binary', 'Prefer not to say'],
              "DA": ['Mand', 'Kvinde', 'Non-binær', 'Ønsker ikke at svare']},
    'EDUCATION': {"EN": ['Primary and lower secondary education (e.g., Folkeskole or Friskole)', 'Upper secondary education (e.g., STX, HTX, HHX, HF)',
                        'Vocational education and training (EUD or EUX)', 'Short-cycle higher education (1-2 years)', 'Medium-length higher education (3-4 years)',
                        'Long-cycle higher education (5-7 years)', 'PhD or other research degree.', 'Other', 'Prefer not to say'],
              "DA": ['Grundskole eller tilsvarende (f.eks. Folkeskole eller Friskole)', 'Gymnasial uddannelse (f.eks. STX, HTX, HHX, HF)',
                    'Erhvervsuddannelse (f.eks. EUD eller EUX)', 'Kort videregående uddannelse (1-2år)',
                    'Mellemlang videregående uddannelse (3–4 år)', 'Lang videregående uddannelse (5–7 år)',
                    'PhD anden forskeruddannelse', 'Andet', 'Ønsker ikke at svare']},
    'JOB': {"EN": ['In paid work (employee, self-employed, working for your family business) or temporary absent.', 'In education (not paid for by employer), even if on vacation',
                  'Unemployed and actively looking for a job', 'Unemployed, wanting a job but not actively looking for a job', 'Permanently sick or disabled',
                  'Retired', 'In community or military service', 'Doing housework, looking after children or other persons', 'Other', 'Prefer not to say'],
              "DA": ['I lønnet arbejde (ansat, selvstændig eller i familiens virksomhed), også midlertidigt fraværende', 
                     'Under uddannelse (ikke betalt af arbejdsgiver), også hvis du har haft ferie', 'Arbejdsløs og aktivt jobsøgende',	
                     'Arbejdsløs, ønsker job, søger ikke aktivt', 'Varigt syg eller med handicap', 'Pensioneret',
                     'I samfunds- eller militærtjeneste', 'Hjemmegående/passer børn eller andre personer', 'Andet', 'Ønsker ikke at svare']},
    'SRQV1': {"EN": ['It should only be allowed to give one vote to each candidate.', 'It should be allowed to give maximally two or three votes to one.',
              'It should be allowed to give more votes to one candidate, but an additional vote should "cost" more than one vote.', 
              'It should be allowed to give all votes to just one candidate without additional "costs".'],
              "DA": ['Det bør kun være tilladt at give én stemme til hver kandidat.', 'Det bør være tilladt at give højst to eller tre stemmer til én kandidat.',
                     'Det bør være tilladt at give flere stemmer til én kandidat, men en ekstra stemme bør "koste" mere end én stemme.',
                     'Det bør være tilladt at give alle stemmer til én kandidat uden ekstra "omkostninger".']}
}

In [7]:
survey_df_dict = {col_nam: [] for col_nam in column_name_list}

for res in survey_full_results:
    for col_nam in column_name_list:
        if "ATTN_CHK_" in col_nam:
            survey_df_dict[col_nam].append(res[col_nam])
        elif col_nam == 'SRQV1':
            survey_df_dict[col_nam].append(res['srqv1'])
        else:
            if col_nam in res['surveyData'].keys():
                survey_df_dict[col_nam].append(res['surveyData'][col_nam])
            else:
                for key in res['surveyData'].keys():
                    if type(res['surveyData'][key]) is dict and col_nam in list(res['surveyData'][key].keys()):
                        survey_df_dict[col_nam].append(res['surveyData'][key][col_nam])

In [8]:
# Translate survey responses to English at the item level.
# Important: do NOT use respondent-level language as the condition for translation.
# Some respondents have mixed-language values, e.g. English ATTN_CHK but Danish ALCL11.
# The rule below is: Danish labels are translated; English labels stay unchanged;
# blanks/NaN/unexpected free-text values are preserved.

def _norm_label(x):
    """Normalize unicode and whitespace for safer dictionary matching."""
    if not isinstance(x, str):
        return x
    return " ".join(unicodedata.normalize("NFKC", x).split())


def _make_translation_maps(translation_dict):
    translation_maps = {}
    for col, langs in translation_dict.items():
        da_vals = langs["DA"]
        en_vals = langs["EN"]

        if len(da_vals) != len(en_vals):
            raise ValueError(
                f"Translation list length mismatch for {col}: "
                f"{len(da_vals)} DA values vs {len(en_vals)} EN values"
            )

        mapping = {}
        for da, en in zip(da_vals, en_vals):
            # DA -> EN
            mapping[da] = en
            mapping[_norm_label(da)] = en
            # EN -> EN, so the translation is safe to run repeatedly
            mapping[en] = en
            mapping[_norm_label(en)] = en

        translation_maps[col] = mapping
    return translation_maps


def _translate_value(value, mapping):
    if isinstance(value, list):
        return [_translate_value(v, mapping) for v in value]
    if pd.isna(value) or value == "":
        return value
    return mapping.get(value, mapping.get(_norm_label(value), value))


survey_df_dict_ENG = deepcopy(survey_df_dict)
translation_maps = _make_translation_maps(translation_dict)

for trans_col_nam, mapping in translation_maps.items():
    survey_df_dict_ENG[trans_col_nam] = [
        _translate_value(value, mapping)
        for value in survey_df_dict_ENG[trans_col_nam]
    ]

# QC: this should be empty if all dictionary-covered Danish labels were translated.
remaining_danish_labels = {}
for trans_col_nam, langs in translation_dict.items():
    da_labels = set(langs["DA"]) | {_norm_label(x) for x in langs["DA"]}
    bad_rows = []
    for i, value in enumerate(survey_df_dict_ENG[trans_col_nam]):
        values = value if isinstance(value, list) else [value]
        if any(isinstance(v, str) and (v in da_labels or _norm_label(v) in da_labels) for v in values):
            bad_rows.append(i)
    if bad_rows:
        remaining_danish_labels[trans_col_nam] = bad_rows

print("Remaining Danish dictionary labels after translation:", remaining_danish_labels)


Remaining Danish dictionary labels after translation: {}


In [9]:
# Create analysis-ready dummy variables for multi-select questions.
# This keeps the original list columns (ALCL11 and JOB), but adds binary indicators.

def _norm_dummy_label(x):
    """Normalize labels so dummies are robust to whitespace and quote variants."""
    if not isinstance(x, str):
        return x
    x = unicodedata.normalize("NFKC", x)
    x = x.replace("’", "'").replace("‘", "'")
    x = x.replace("“", '"').replace("”", '"')
    x = " ".join(x.split())
    return x


def _as_list(value):
    """Return a list for list-like survey answers, preserving safe behavior for blanks."""
    if isinstance(value, list):
        return value
    if value is None:
        return []
    try:
        if pd.isna(value):
            return []
    except Exception:
        pass
    if isinstance(value, str):
        value = value.strip()
        if value == "":
            return []
        if value.startswith("[") and value.endswith("]"):
            try:
                parsed = ast.literal_eval(value)
                return parsed if isinstance(parsed, list) else [parsed]
            except Exception:
                return [value]
    return [value]


def _add_multiselect_dummies(data_dict, source_col, dummy_map):
    """Add 0/1 dummies to data_dict based on whether each label is selected."""
    normalized_dummy_map = {
        dummy_col: _norm_dummy_label(label)
        for dummy_col, label in dummy_map.items()
    }

    selected_sets = [
        {_norm_dummy_label(v) for v in _as_list(value)}
        for value in data_dict[source_col]
    ]

    for dummy_col, normalized_label in normalized_dummy_map.items():
        data_dict[dummy_col] = [
            int(normalized_label in selected)
            for selected in selected_sets
        ]


# ALCL11: important local issue selections -> ALCL1_imp ... ALCL10_imp
ALCL11_dummy_map = {
    f"ALCL{i}_imp": label
    for i, label in enumerate(translation_dict["ALCL11"]["EN"], start=1)
}
_add_multiselect_dummies(survey_df_dict_ENG, "ALCL11", ALCL11_dummy_map)


# JOB: employment-status multi-select question -> stable job-status dummy variables
JOB_dummy_map = {
    "JOB_paid_work": "In paid work (employee, self-employed, working for your family business) or temporary absent.",
    "JOB_education": "In education (not paid for by employer), even if on vacation",
    "JOB_unemployed_looking": "Unemployed and actively looking for a job",
    "JOB_unemployed_not_looking": "Unemployed, wanting a job but not actively looking for a job",
    "JOB_permanently_sick_disabled": "Permanently sick or disabled",
    "JOB_retired": "Retired",
    "JOB_community_military_service": "In community or military service",
    "JOB_housework_care": "Doing housework, looking after children or other persons",
    "JOB_other": "Other",
    "JOB_prefer_not_to_say": "Prefer not to say",
}
_add_multiselect_dummies(survey_df_dict_ENG, "JOB", JOB_dummy_map)

print("Added ALCL11 dummy columns:", list(ALCL11_dummy_map.keys()))
print("Added JOB dummy columns:", list(JOB_dummy_map.keys()))

Added ALCL11 dummy columns: ['ALCL1_imp', 'ALCL2_imp', 'ALCL3_imp', 'ALCL4_imp', 'ALCL5_imp', 'ALCL6_imp', 'ALCL7_imp', 'ALCL8_imp', 'ALCL9_imp', 'ALCL10_imp']
Added JOB dummy columns: ['JOB_paid_work', 'JOB_education', 'JOB_unemployed_looking', 'JOB_unemployed_not_looking', 'JOB_permanently_sick_disabled', 'JOB_retired', 'JOB_community_military_service', 'JOB_housework_care', 'JOB_other', 'JOB_prefer_not_to_say']


In [10]:
df_survey_response = pd.DataFrame.from_dict(survey_df_dict); df_survey_response

,userId,surveyDuration,AGE,SCREEN_ELIGIBILITY,SCREEN_RESIDENCE,NSEC_B,NSEC_BCRT,NSEC_P,NECN_B,NECN_BCRT,...,SSBS1,SSBS2,SQDR1,GENDER,EDUCATION,JOB,SRQV1,language,ATTN_CHK_PF,ATTN_CHK_FAIL_REASON
0,3a203770e837d765cd5f6f984a4f598c,406.388,57,Ja,Ja,7,6,5,6,6,...,7,5,5,Kvinde,PhD anden forskeruddannelse,"[I lønnet arbejde (ansat, selvstændig eller i ...",Det bør være tilladt at give alle stemmer til ...,DA,FAIL,Zero correct matches
1,3a203770e80d24440b6379c848d401f9,463.966,77,Ja,Ja,7,8,8,10,10,...,10,10,0,Mand,Erhvervsuddannelse (f.eks. EUD eller EUX),[Pensioneret],Det bør kun være tilladt at give én stemme til...,DA,FAIL,Zero correct matches
2,3a203770e7e805db58c55016db059137,859.743,41,Ja,Ja,6,10,2,4,10,...,5,6,6,Mand,Erhvervsuddannelse (f.eks. EUD eller EUX),"[I lønnet arbejde (ansat, selvstændig eller i ...",Det bør kun være tilladt at give én stemme til...,DA,FAIL,Zero correct matches
3,3a203770e848ec257fce0316dd4cfd65,276.418,48,Ja,Ja,7,9,9,6,9,...,10,10,0,Kvinde,Erhvervsuddannelse (f.eks. EUD eller EUX),"[I lønnet arbejde (ansat, selvstændig eller i ...",It should be allowed to give all votes to just...,DA,PASS,None
4,3a203770e7efbe8b69127c396a4165f4,341.944,44,Ja,Ja,10,10,0,9,10,...,10,9,1,Mand,Mellemlang videregående uddannelse (3–4 år),"[I lønnet arbejde (ansat, selvstændig eller i ...",Det bør kun være tilladt at give én stemme til...,DA,PASS,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1014,3a203c634ef0953007536acb9b60f568,624.978,27,Ja,Ja,8,6,5,6,5,...,6,5,5,Kvinde,Mellemlang videregående uddannelse (3–4 år),"[I lønnet arbejde (ansat, selvstændig eller i ...",Det bør være tilladt at give alle stemmer til ...,DA,FAIL,Zero correct matches
1015,3a2054ba33aaf5d31419ee397ce605f6,1805.128,34,Ja,Ja,6,5,4,10,10,...,7,7,3,Kvinde,Mellemlang videregående uddannelse (3–4 år),"[I lønnet arbejde (ansat, selvstændig eller i ...",Det bør være tilladt at give flere stemmer til...,DA,PASS,None
1016,3a20644a9722a8e51052ba2f2ff0aeff,396.447,34,Ja,Ja,6,5,3,3,5,...,7,4,5,Mand,Lang videregående uddannelse (5–7 år),"[I lønnet arbejde (ansat, selvstændig eller i ...",It should only be allowed to give one vote to ...,DA,PASS,None
1017,3a20644a97f8894830a72b2fd9f4c03e,707.764,23,Ja,Ja,4,6,6,3,6,...,8,8,8,Mand,"Gymnasial uddannelse (f.eks. STX, HTX, HHX, HF)",[Ønsker ikke at svare],Det bør være tilladt at give højst to eller tr...,DA,PASS,None


In [11]:
df_survey_response_ENG = pd.DataFrame.from_dict(survey_df_dict_ENG); df_survey_response_ENG

,userId,surveyDuration,AGE,SCREEN_ELIGIBILITY,SCREEN_RESIDENCE,NSEC_B,NSEC_BCRT,NSEC_P,NECN_B,NECN_BCRT,...,JOB_paid_work,JOB_education,JOB_unemployed_looking,JOB_unemployed_not_looking,JOB_permanently_sick_disabled,JOB_retired,JOB_community_military_service,JOB_housework_care,JOB_other,JOB_prefer_not_to_say
0,3a203770e837d765cd5f6f984a4f598c,406.388,57,Yes,Yes,7,6,5,6,6,...,1,0,0,0,0,0,0,0,0,0
1,3a203770e80d24440b6379c848d401f9,463.966,77,Yes,Yes,7,8,8,10,10,...,0,0,0,0,0,1,0,0,0,0
2,3a203770e7e805db58c55016db059137,859.743,41,Yes,Yes,6,10,2,4,10,...,1,0,0,0,0,0,0,0,0,0
3,3a203770e848ec257fce0316dd4cfd65,276.418,48,Yes,Yes,7,9,9,6,9,...,1,0,0,0,0,0,0,0,0,0
4,3a203770e7efbe8b69127c396a4165f4,341.944,44,Yes,Yes,10,10,0,9,10,...,1,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1014,3a203c634ef0953007536acb9b60f568,624.978,27,Yes,Yes,8,6,5,6,5,...,1,0,0,0,0,0,0,0,0,0
1015,3a2054ba33aaf5d31419ee397ce605f6,1805.128,34,Yes,Yes,6,5,4,10,10,...,1,0,0,0,0,0,0,0,0,0
1016,3a20644a9722a8e51052ba2f2ff0aeff,396.447,34,Yes,Yes,6,5,3,3,5,...,1,0,0,0,0,0,0,0,0,0
1017,3a20644a97f8894830a72b2fd9f4c03e,707.764,23,Yes,Yes,4,6,6,3,6,...,0,0,0,0,0,0,0,0,0,1


In [12]:
df_survey_response.to_csv('survey_responses_original.csv', encoding='utf-8', index=False)
df_survey_response_ENG.to_csv('survey_responses_ENG_TRANS.csv', encoding='utf-8', index=False)
